# Sampling-based model predictive control simulation
#### Payload type: GPU tensor computation + MuJoCo simulation + rendering

In [ ]:
# Imports
import gymnasium as gym
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import animation, rc
from ipywidgets import interact, widgets
# Change to glfw when on Mac. This might fuck up the lighting.
%env MUJOCO_GL=egl
import mujoco
import mujoco.viewer
import mediapy as media

import torch
from torch import nn
import torch.nn.functional as F
from tensordict import TensorDict
from torchrl.data import CompositeSpec, BoundedTensorSpec, UnboundedContinuousTensorSpec
from torchrl.envs.model_based import ModelBasedEnvBase
from torchrl.modules import SafeModule
from torchrl.modules import CEMPlanner
from torchrl.modules import WorldModelWrapper
from torchrl.data import TensorDictReplayBuffer

In [ ]:
class MujocoCartPoleEnv(gym.Env):
    def __init__(self, m = 2, M = 5, l = 0.5, g = 9.81, k = 100, dt = 0.02, end_on_failure = True):
        self.model = self._get_mj_model(m,M,l,g,k,dt)
        self.data = mujoco.MjData(self.model)
        self.renderer = mujoco.Renderer(self.model)
        self.truncate = 1000
        self.action_space = gym.spaces.Box(low=-1, high=1, shape=(self.model.nu,))
        self.observation_space = gym.spaces.Box(low=-np.inf, high=np.inf, shape=(self.model.nq + self.model.nv,))
        self.end_on_failure = end_on_failure
        self.reset()

    def reset(self):
        mujoco.mj_resetData(self.model, self.data)
        self.data.qvel = 0.0*np.random.randn(2) # velocities
        self.data.qpos = 0.1*np.random.randn(2) # positions
        self.timestep = 0
        return self.get_state(), {}

    def set_state(self, state):
        self.data.qpos = state[:self.model.nq]
        self.data.qvel = state[self.model.nq:]
    
    def get_state(self):
        return np.concatenate([self.data.qpos, self.data.qvel])

    def step(self, action):
        self.data.ctrl = action
        mujoco.mj_step(self.model, self.data)
        next_state = self.get_state()
        reward = self._calculate_reward()
        terminal = self._is_done()
        truncated = self.timestep >= self.truncate
        info = {}  # Additional information (optional)
        self.timestep += 1
        return (next_state, reward, terminal, truncated, info)

    def render(self):
        self.renderer.update_scene(self.data)
        return self.renderer.render()

    def _calculate_reward(self):
        # Calculate reward based on the current state (optional)
        return int(self.data.qpos[1] > - np.pi/12 and self.data.qpos[1] < np.pi/12)

    def _is_done(self):
        # Check if the episode is done based on the current state (optional)
        if self.end_on_failure:
            return (self.data.qpos[1] < - np.pi/12 or self.data.qpos[1] > np.pi/12)
        return False

    # return a mujoco model with the given physical parameters
    @staticmethod
    def _get_mj_model(m,M,l,g,k,dt):
        xml = f"""
        <mujoco model='test_cartpole'>
            <compiler inertiafromgeom='true' coordinate='local'/>

            <size nkey="1"/>

            <option timestep='{dt}' integrator="RK4" gravity='0 0 {-g}'/>

            <default>
            <joint damping='0.0' solreflimit='.08 1'/>
            <geom contype='0' friction='0. 0. 0.'/>
            </default>

            <worldbody>
            <camera name='fixed' pos='0 -2.5 0' quat='0.707 0.707 0 0'/>
            <light name="top" castshadow="false"/>
            <geom name='floor' pos='0 0 -1' size='4 4 4' type='plane' />
            <geom name='rail1' type='capsule' pos='0 .07 0' quat='0.707 0 0.707 0'
                    size='0.02 2.2' />
            <geom name='rail2' type='capsule' pos='0 -.07 0' quat='0.707 0 0.707 0'
                    size='0.02 2.2' />
            <body name='cart' pos='0 0 0'>
                <camera name='cart' pos='0 -2.5 0' quat='0.707 0.707 0 0' />
                <joint name='slider' type='slide' limited='true' pos='0 0 0'
                        axis='1 0 0' range='-2 2' />
                <geom name='cart' type='box' pos='0 0 0'
                        mass='{M}' size='0.2 0.1 0.05' rgba='0.7 0.7 0 1' />
                <site name='cart sensor' type='box' pos='0 0 0'
                        size='0.2 0.1 0.05' rgba='0.7 0.7 0 0' />
                <body name='pole' pos='0 0 0'>
                <camera name='pole'  pos='0 -2.5 0' quat='0.707 0.707 0 0' />
                <joint name='hinge' type='hinge' pos='0 0 0' axis='0 1 0'/>
                <geom name='cpole' type='capsule' fromto='0 0 0 0 0 {l}'
                        mass='0' size='0.01 {l}' rgba='0 0.7 0.7 1' />
                <geom type='sphere' size='.05' name='tip' mass='{m}' pos='.001 0 {l}'/>
                </body>
            </body>
            </worldbody>

            <actuator>
            <motor name='slide' joint='slider' gear='{k}' ctrllimited='true' ctrlrange='-1 1' />
            </actuator>

        </mujoco>
        """
        return mujoco.MjModel.from_xml_string(xml)

In [ ]:
def render_episode(policy, env):
	frames = []
	score = 0
	state = env.get_state()
	terminated = truncated = False
	while not terminated and not truncated:
		frame = env.render()
		frames.append(frame)
		action = policy(state)
		state, reward, terminated, truncated, _ = env.step(action)
		score += reward     
	env.close()
	print(f"Finished episode. Cummulative Return: {score}")
	media.show_video(frames, fps=100, loop=True)

In [ ]:
# Forward model that takes in a state and an action to predict the next action using the non-linear equations of motion that we derived
# in the control exercise. The model uses a simpler integration method than MuJoCo, resulting in different trajectories.
def predict_next_states(states, actions):
	m = 2
	M = 5
	l = 0.5
	g = 9.81
	k = 100
	dt = 0.02
	xs, thetas, xs_dot, thetas_dot = states.T
	actions = actions.squeeze()

	# Non-linear equations of motion for the cart-pole system
	def equations_of_motion(xs, thetas, xs_dot, thetas_dot, actions):
		sin_thetas = torch.sin(thetas)
		cos_thetas = torch.cos(thetas)
		xs_dot_dot = (k * actions + m * l * thetas_dot**2 * sin_thetas - m * g * sin_thetas * cos_thetas) / (M + m - m * cos_thetas**2)
		thetas_dot_dot = 1/l * (g * sin_thetas - cos_thetas * xs_dot_dot) 
		return xs_dot, thetas_dot, xs_dot_dot, thetas_dot_dot

	# Integrate the equations of motion using simple Euler's method
	xs_dot, thetas_dot, xs_dot_dot, thetas_dot_dot = equations_of_motion(xs, thetas, xs_dot, thetas_dot, actions)
	xs_dot += xs_dot_dot * dt
	thetas_dot += thetas_dot_dot * dt
	xs += xs_dot * dt
	thetas += thetas_dot * dt

	return torch.stack([xs, thetas, xs_dot, thetas_dot]).T

In [ ]:
# Objective function
def quadratic_cost(trajectory, desired_state = torch.zeros(4), weight = torch.tensor([1.,2.,0.,0.])):
	with torch.no_grad():
		return torch.square(trajectory-desired_state)@weight

In [ ]:
# predict a state trajectory given an initial state and a sequence of actions, using the given dynamics model
def predict_trajectories(states, action_sequences, dynamics_model):
    trajectories = torch.zeros((len(action_sequences), len(action_sequences[0]), len(states[0])), device=states.device)
    for i, actions in enumerate(action_sequences.T):
        states = dynamics_model(states, actions)
        trajectories[:,i,:] = states
    return trajectories

In [ ]:
# predict a trajectory and evaluate it using the given objective function
def evaluate_action_sequences(state, action_sequences, desired_state=torch.zeros(4), dynamics_model=predict_next_states, objective_function=quadratic_cost, cost_weights = torch.tensor([1.,2.,0.,0.])):
    trajectories= predict_trajectories(torch.tile(state,(len(action_sequences),1)), action_sequences, dynamics_model)
    return objective_function(trajectories, desired_state, cost_weights).sum(dim=1)

In [ ]:
## Solution
# Sampling-Based Planner
def plan(state, desired_state, dynamics_model, objective_function, cost_weights, horizon = 50, num_candidates=20000):
	action_sequences = torch.distributions.uniform.Uniform(-1,1).sample((num_candidates,horizon)).to(state.device)
	costs = evaluate_action_sequences(state, action_sequences, desired_state, dynamics_model, objective_function, cost_weights)
	best_cand = costs.argmin()
	return action_sequences[best_cand,0]

In [ ]:
# Simulate the cart-pole system with MPC
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
desired_state = torch.zeros(4).to(device)
cost_weights = torch.tensor([1.,2.,0.1,0.1]).to(device)
env = MujocoCartPoleEnv(end_on_failure=False)
env.reset()
render_episode(lambda s: plan(torch.tensor(s).to(device), desired_state, predict_next_states, quadratic_cost, cost_weights).cpu().numpy(), env)